# Explainability — Grad-CAM and Perona-Malik grid maps

This notebook visualises **why** the federated ResNet18 predicts each class.

- **Grad-CAM** (target: `layer4`, final residual stage) highlights image regions
  that most influenced the classifier logit for the predicted class.
- **Perona-Malik grid map** uses the **same `feature_layer` and `grid_size`** as
  PIDL training (default `layer2`, grid 4×4). Each cell score is the mean squared
  PM residual in that region — strong values indicate pronounced edge/texture
  structure under the physics-inspired operator.
- **Overlap (IoU, top 25%)** compares where Grad-CAM and the PM map agree.
  Agreement suggests predictions align with physics-guided spatial features.

**Modes:** if `final_model.pth` exists under `results_explainability/...`, weights
are loaded; otherwise the full Flower FL pipeline is re-run and the global model
is saved.


---
## A. Setup

Optional Google Drive mount (skip if you store the repo only under `/content`).


In [2]:
# Optional: Google Drive (uncomment if you keep data or repo on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, json, gc, time
from pathlib import Path

GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'
PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
    if not PROJECT_ROOT.exists():
        raise RuntimeError('Clone failed — set GITHUB_REPO to your fork.')
else:
    os.system(f'git -C {PROJECT_ROOT} pull')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.7 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5,

---
## A (continued). Experiment configuration

Edit `dataset_name` and `KAGGLE_SLUGS` mapping as needed. FL hyperparameters should
match training you want to explain.


In [3]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ── Dataset selection (edit slug in one place) ────────────────────
dataset_name = 'brain_tumor_mri'   # brain_tumor_mri | colon_cancer_or_pathology | covid

KAGGLE_SLUGS = {
    'brain_tumor_mri':          'masoudnickparvar/brain-tumor-mri-dataset',
    'colon_cancer_or_pathology': 'andrewmvd/lung-and-colon-cancer-histopathological-images',
    'covid':                    'tawsifurrahman/covid19-radiography-database',
}
COLON_OR_LUNG = 'colon_image_sets'  # for colon dataset only

# ── Federated training (match main experiments) ───────────────────
num_clients     = 3
num_rounds      = 5
local_epochs    = 2
batch_size      = 32
learning_rate   = 1e-3
image_size      = 224
random_seed     = 42
feature_layer   = 'layer2'
grid_size       = 4
lambda_pm       = 0.10
regularizer_type = 'perona_malik'
use_grid_loss   = True
k_pm            = 1.0
use_secagg      = False   # SecAgg+ off in run_simulation (same as notebooks 01/04)

# ── Explainability ────────────────────────────────────────────────
samples_per_class = 5
gradcam_layer     = 'layer4'   # final conv stage for Grad-CAM

RESULT_DIR = PROJECT_ROOT / 'results_explainability' / dataset_name / f'{num_clients}_clients'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG = RESULT_DIR / 'figures'
(FIG / 'gradcam').mkdir(parents=True, exist_ok=True)
(FIG / 'pm_grid').mkdir(parents=True, exist_ok=True)
(FIG / 'combined').mkdir(parents=True, exist_ok=True)
(FIG / 'summary').mkdir(parents=True, exist_ok=True)

print('Results ->', RESULT_DIR)


Results -> /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients


---
## B. Download data and resolve ImageFolder root


In [4]:
import kagglehub
from data.kaggle_loader import find_image_root

slug = KAGGLE_SLUGS[dataset_name]
raw = kagglehub.dataset_download(slug)
print('Downloaded:', raw)

if dataset_name == 'colon_cancer_or_pathology':
    cand = list(Path(raw).rglob(COLON_OR_LUNG))
    data_root = str(cand[0]) if cand else str(find_image_root(raw))
else:
    data_root = str(find_image_root(raw))

print('image_root:', data_root)


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Downloaded: /kaggle/input/brain-tumor-mri-dataset
[find_image_root] Found (named training split): 'Training'
  Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
image_root: /kaggle/input/brain-tumor-mri-dataset/Training


---
## C. Model handling — load `final_model.pth` or retrain FL


In [5]:
model_path = RESULT_DIR / 'final_model.pth'
need_train = not model_path.exists()
print('final_model.pth exists:', not need_train)

if need_train:
    import json as _json
    import gc as _gc
    from flwr.simulation import run_simulation
    from federated.server_app import app as _server_app
    from federated.client_app import _make_client_app

    run_cfg = {
        'dataset_name': dataset_name,
        'data_root': data_root,
        'log_dir': str(RESULT_DIR),
        'num_classes': 0,
        'num_clients': num_clients,
        'num_server_rounds': num_rounds,
        'min_fit_clients': num_clients,
        'local_epochs': local_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'image_size': image_size,
        'feature_layer': feature_layer,
        'regularizer_type': regularizer_type,
        'lambda_pm': lambda_pm,
        'use_grid_loss': use_grid_loss,
        'grid_size': grid_size,
        'k': k_pm,
        'random_seed': random_seed,
        'use_secagg': use_secagg,
        'enable_attack': False,
        'enable_update_clipping': False,
    }
    os.environ['FL_RUN_OVERRIDE'] = _json.dumps(run_cfg)
    _client_app = _make_client_app()
    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    t0 = time.time()
    run_simulation(
        server_app=_server_app,
        client_app=_client_app,
        num_supernodes=num_clients,
        backend_config={'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}},
    )
    print(f'FL done in {time.time()-t0:.0f}s')
    try:
        from federated.server_app import finalize_experiment
        finalize_experiment()
    except Exception as exc:
        print('finalize_experiment:', exc)
    os.environ.pop('FL_RUN_OVERRIDE', None)
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert model_path.exists(), 'final_model.pth missing after training'
print('Using weights:', model_path)


final_model.pth exists: False



            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 2, 'num_gpus': 0.5}}
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
DEBUG:flwr:Asyncio event loop al

[Server] Device: cuda  |  Log dir: /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients
[task] Building federated data for 3 clients from: /kaggle/input/brain-tumor-mri-dataset/Training
[build_federated_dataloaders] Loading ImageFolder from '/kaggle/input/brain-tumor-mri-dataset/Training' …


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  → 5,600 images  |  4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
  → Grayscale source detected. Images will be converted to 3-channel RGB.
  → Split: 4,480 train  |  1,120 test  (stratified, seed=42)
  → Partitioning (iid): client 0: 1,496  |  client 1: 1,492  |  client 2: 1,492
[dataset_utils] Summary saved → /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients/dataset_summary.json
[build_federated_dataloaders] Done. 3 client loaders + 1 test loader ready.

[Server] Dataset  : brain_tumor_mri  (4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary'])
[Server] Test set : 1120 samples


...

Round   5 | Train Acc: 91.83%  Loss: 0.2455  PIDL: 0.032025 | Client Val Acc: 95.80%  Loss: 0.1194 |  Server Acc: 95.80% | Elapsed: 134.7s


(ClientAppActor pid=4844) WARNING :   Manually terminating ClientAppActor


FL done in 756s

  EXPERIMENT SUMMARY
  Dataset:  brain_tumor_mri   Clients: 3   Rounds: 6
  Best Accuracy             0.9580  (round 5)
  Final Accuracy            0.9580
  Best Balanced Acc         0.9580  (round 5)
  Final Balanced Acc        0.9580
  Best Macro F1             0.9579  (round 5)
  Final Macro F1            0.9579
  Best ROC-AUC              0.9969  (round 5)
  Best PR-AUC               0.9909  (round 5)
  Final ECE                 0.0164
  Train time (total)        0.0000
  Infer time (total)        53.5200


[ExperimentLogger] All outputs written to: /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients
  fl_rounds.csv        (6 rounds)
  per_class_metrics.csv
  fl_eval.json
  fl_summary.json


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[finalize_experiment] Final model saved -> /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients/final_model.pth
Using weights: /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients/final_model.pth


---
## D. Build global test loader & select balanced samples


In [6]:
from federated.task import get_federated_data
from configs.experiment_config import ModelConfig
from models.resnet_pidl import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = get_federated_data(
    data_root=data_root,
    num_clients=num_clients,
    batch_size=batch_size,
    image_size=image_size,
    random_seed=random_seed,
    save_summary_to=str(RESULT_DIR),
)
num_classes = data['num_classes']
class_names = list(data['class_names'])
test_loader = data['global_test_loader']

# Navigate: TransformDataset -> Subset -> ImageFolder
tds = test_loader.dataset
sub = tds.subset
base_folder = sub.dataset
subset_indices = list(sub.indices)

by_class = {c: [] for c in range(num_classes)}
for j in range(len(sub)):
    idx = subset_indices[j]
    y = int(base_folder.samples[idx][1])
    by_class[y].append(j)

rng = random.Random(random_seed)
picked = []
for c in range(num_classes):
    pool = by_class[c][:]
    rng.shuffle(pool)
    for j in pool[:samples_per_class]:
        idx = subset_indices[j]
        path = base_folder.samples[idx][0]
        picked.append({'ds_index': j, 'true_class': c, 'class_name': class_names[c], 'path': path})

pd.DataFrame(picked).to_csv(RESULT_DIR / 'selected_samples.csv', index=False)
print(f'Selected {len(picked)} samples')


Selected 20 samples


---
## E–G. Run Grad-CAM, PM grid maps, combined figures, CSV log


In [7]:
import matplotlib.pyplot as plt

from explainability.gradcam import GradCAM, cam_statistics, upsample_cam
from explainability.pm_grid_explainer import (
    gradcam_pm_iou_top25,
    grid_statistics,
    normalize01,
    pm_grid_scores,
    upsample_grid_map,
)
from explainability.plot_utils import overlay_heatmap, savefig_tight, tensor_to_display_rgb

model_cfg = ModelConfig(pidl_feature_layer=feature_layer)  # type: ignore[arg-type]
model = build_model(num_classes=num_classes, config=model_cfg)
sd = torch.load(model_path, map_location='cpu')
model.load_state_dict(sd)
model.to(device)
model.eval()

target_mod = getattr(model, gradcam_layer)

rows = []
y_true_sel = []
y_pred_sel = []

for i, rec in enumerate(picked):
    image_id = f'{i:04d}'
    x, _ = tds[rec['ds_index']]
    x = x.unsqueeze(0).to(device)
    H = W = image_size

    with GradCAM(model, target_mod) as gcam:
        cam_hw, logits, pred_idx = gcam.compute(x, class_idx=None)

    prob = torch.softmax(logits, dim=-1)[0, pred_idx].item()
    true_idx = rec['true_class']
    ok = pred_idx == true_idx
    y_true_sel.append(true_idx)
    y_pred_sel.append(pred_idx)

    cam_up = upsample_cam(cam_hw, H, W)
    mean_gc, max_gc = cam_statistics(cam_hw)
    model.zero_grad(set_to_none=True)

    with torch.no_grad():
        _, feats = model(x, return_features=True)
        fm_pidl = feats[feature_layer]

    gs = pm_grid_scores(fm_pidl, grid_size=grid_size, k=k_pm, normalize=True)
    gs_n = normalize01(gs)[0]
    pm_up = upsample_grid_map(gs_n.unsqueeze(0), H, W)
    mean_pm, max_pm, top_cell = grid_statistics(gs_n.unsqueeze(0))

    iou_25 = gradcam_pm_iou_top25(cam_up, pm_up)

    rgb = tensor_to_display_rgb(x[0], (H, W))
    o_gc = overlay_heatmap(rgb, cam_up)
    o_pm = overlay_heatmap(rgb, pm_up, cmap='viridis')

    title = (
        f"true={rec['class_name']} | pred={class_names[pred_idx]} | conf={prob:.3f} | ok={ok}"
    )

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(o_gc)
    ax.set_title('Grad-CAM\n' + title, fontsize=8)
    ax.axis('off')
    savefig_tight(FIG / 'gradcam' / f'{image_id}_gradcam.png')

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(o_pm)
    ax.set_title('PM grid\n' + title, fontsize=8)
    ax.axis('off')
    savefig_tight(FIG / 'pm_grid' / f'{image_id}_pmgrid.png')

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    axes[0].imshow(rgb)
    axes[0].set_title('Input')
    axes[1].imshow(o_gc)
    axes[1].set_title('Grad-CAM')
    axes[2].imshow(o_pm)
    axes[2].set_title('PM grid')
    axes[3].imshow(np.concatenate([o_gc, o_pm], axis=1))
    axes[3].set_title('GC | PM')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, fontsize=9)
    savefig_tight(FIG / 'combined' / f'{image_id}_combined.png')

    rows.append({
        'dataset_name': dataset_name,
        'num_clients': num_clients,
        'image_id': image_id,
        'true_class': rec['class_name'],
        'predicted_class': class_names[pred_idx],
        'confidence': round(prob, 6),
        'correct': ok,
        'mean_gradcam_score': round(mean_gc, 6),
        'max_gradcam_score': round(max_gc, 6),
        'mean_pm_grid_score': round(mean_pm, 6),
        'max_pm_grid_score': round(max_pm, 6),
        'top_pm_grid_index': top_cell,
        'gradcam_pm_overlap_score': round(iou_25, 6),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULT_DIR / 'explainability_summary.csv', index=False)
print(summary_df.head())


      dataset_name  num_clients image_id true_class predicted_class  \
0  brain_tumor_mri            3     0000     glioma          glioma   
1  brain_tumor_mri            3     0001     glioma          glioma   
2  brain_tumor_mri            3     0002     glioma          glioma   
3  brain_tumor_mri            3     0003     glioma          glioma   
4  brain_tumor_mri            3     0004     glioma          glioma   

   confidence  correct  mean_gradcam_score  max_gradcam_score  \
0    0.998509     True            0.317717                1.0   
1    0.999893     True            0.285250                1.0   
2    0.999672     True            0.371918                1.0   
3    0.999853     True            0.357019                1.0   
4    0.999939     True            0.312406                1.0   

   mean_pm_grid_score  max_pm_grid_score  top_pm_grid_index  \
0            0.443006                1.0                  6   
1            0.380053                1.0                

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## H. Save config snapshot


In [8]:
cfg_out = {
    'dataset_name': dataset_name,
    'data_root': data_root,
    'num_clients': num_clients,
    'num_rounds': num_rounds,
    'local_epochs': local_epochs,
    'batch_size': batch_size,
    'image_size': image_size,
    'feature_layer': feature_layer,
    'grid_size': grid_size,
    'lambda_pm': lambda_pm,
    'k_pm': k_pm,
    'gradcam_layer': gradcam_layer,
    'samples_per_class': samples_per_class,
    'random_seed': random_seed,
}
(RESULT_DIR / 'explainability_config.json').write_text(json.dumps(cfg_out, indent=2))


363

---
## I. Summary plots and tables


In [9]:
# Mean overlap & confidence by true class
by_cls = summary_df.groupby('true_class').agg(
    gradcam_pm_overlap=('gradcam_pm_overlap_score', 'mean'),
    confidence=('confidence', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_cls['true_class'], by_cls['gradcam_pm_overlap'], color='steelblue')
ax.set_ylabel('Mean Grad-CAM / PM IoU (top 25%)')
ax.set_title('Overlap by true class')
plt.xticks(rotation=30, ha='right')
savefig_tight(FIG / 'summary' / 'bar_overlap_by_class.png')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_cls['true_class'], by_cls['confidence'], color='darkseagreen')
ax.set_ylabel('Mean confidence')
ax.set_title('Confidence by true class (selected samples)')
plt.xticks(rotation=30, ha='right')
savefig_tight(FIG / 'summary' / 'bar_confidence_by_class.png')

from IPython.display import display
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true_sel, y_pred_sel, labels=list(range(num_classes)))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.colorbar(im, ax=ax)
fig.suptitle('Confusion — selected samples only')
savefig_tight(FIG / 'summary' / 'confusion_selected.png')

correct_tbl = summary_df.groupby('correct').size().rename('count').reset_index()
correct_tbl.to_csv(RESULT_DIR / 'explanation_correctness_table.csv', index=False)
display(correct_tbl)


,correct,count
0,False,1
1,True,19


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## Download (Colab)


In [10]:
import subprocess
zip_path = '/content/explainability_results.zip'
subprocess.run(['zip', '-r', zip_path, str(RESULT_DIR)], check=True)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not on Colab — results in', RESULT_DIR)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>